# 99 — Perbandingan Text-only vs Metadata (test set yang sama)

Jalankan setelah keempat notebook training selesai. Taruh semua `preds_*.csv` di `PRED_DIR`.

Yang dihitung:
1. **Level annotator** — tiap baris (teks, annotator) di test set. Text-only memberi prediksi yang sama untuk semua annotator sebuah teks; metadata bisa berbeda per annotator. Acuan = label annotator itu.
2. **Level teks** — prediksi metadata dirata-ratakan per teks, dibandingkan dengan text-only terhadap label majority.
3. **Subset ≥2 annotator**, **subset sengketa** (teks dengan ≥2 annotator yang tidak sepakat), dan baris **minoritas** (annotator yang berbeda dari mayoritas). Ini uji langsung hipotesis.
4. **Per kelompok demografis** — F1 level annotator per nilai atribut (mis. gender = perempuan).
5. **Bootstrap 95% CI** untuk selisih (metadata − text-only), resampling per teks.
6. Kalau `FINAL_SEEDS` > 1: rata-rata ± std antar seed.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

PRED_DIR = "/kaggle/working"          # folder berisi semua preds_*.csv
OUT_FILE = f"{PRED_DIR}/comparison_results.csv"
MODELS   = ["indobertweet", "nusabert"]
POL_TAG  = "nopol"                     # "pol" untuk membandingkan run dengan polarisasi
META_TAG = "jabo"                      # sesuaikan dengan META_TAG di notebook metadata
N_BOOT   = 1000
SEED     = 42

LABEL_COLS = ["insults", "identity_attack", "threat_incitement_to_violence",
              "profanity_obscenity", "sexually_explicit"]
DEMO_COLS = ["domisili_seg", "ethnicity", "religion", "disability", "lgbt", "gender", "age", "domisili",
             "pendidikan terakhir", "status pekerjaan", "president vote leaning"]

def path(model, setup, stage):
    tag = "_".join(x for x in [model, setup, META_TAG if setup == "meta" else "", POL_TAG] if x)
    return f"{PRED_DIR}/preds_{tag}_s{stage}.csv"

def load(model, setup, stage):
    return pd.read_csv(path(model, setup, stage), dtype={"text_id": str})

## Fungsi metrik & bootstrap

In [ ]:
def bin_metrics(y, p):
    return {"f1_toxic": f1_score(y, p, zero_division=0),
            "f1_macro": f1_score(y, p, average="macro", zero_division=0),
            "precision": precision_score(y, p, zero_division=0),
            "recall": recall_score(y, p, zero_division=0),
            "accuracy": accuracy_score(y, p)}

def ml_macro_f1(Y, P):
    return f1_score(Y, P, average="macro", zero_division=0)


def cluster_bootstrap(df, stat_fn, n=N_BOOT, cluster="text_id", seed=SEED):
    """95% CI untuk stat_fn(sub_df), resampling per teks (baris satu teks ikut bersama)."""
    rng = np.random.default_rng(seed)
    groups = list(df.groupby(cluster).indices.values())
    vals = []
    for _ in range(n):
        pick = rng.integers(0, len(groups), len(groups))
        idx = np.concatenate([groups[i] for i in pick])
        vals.append(stat_fn(df.iloc[idx]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return lo, hi


def summarize(name, df, ytrue, pa, pb, metric_fn, key):
    """Bandingkan prediksi a (text-only) vs b (metadata) pada df."""
    a, b = metric_fn(df[ytrue], df[pa])[key], metric_fn(df[ytrue], df[pb])[key]
    lo, hi = cluster_bootstrap(df, lambda d: metric_fn(d[ytrue], d[pb])[key] - metric_fn(d[ytrue], d[pa])[key])
    sig = "ya" if (lo > 0 or hi < 0) else "tidak"
    return {"evaluasi": name, "metrik": key, "n_baris": len(df), "n_teks": df["text_id"].nunique(),
            "text_only": round(a, 4), "metadata": round(b, 4), "selisih": round(b - a, 4),
            "ci_low": round(lo, 4), "ci_high": round(hi, 4), "signifikan": sig}

## Stage 1

In [ ]:
def stage1_frames(model, seed=None):
    t, m = load(model, "text", 1), load(model, "meta", 1)
    if seed is not None:
        t, m = t[t.seed == seed], m[m.seed == seed]
    # rata-rata probabilitas antar seed
    t = t.groupby("text_id").agg(y_true=("y_true", "first"), prob=("prob", "mean")).reset_index()
    keep = {c: "first" for c in DEMO_COLS if c in m.columns}
    m = m.groupby(["text_id", "annotator_id"]).agg(y_true=("y_true", "first"), prob=("prob", "mean"),
                                                   **{c: (c, f) for c, f in keep.items()}).reset_index()

    st, sm = set(t.text_id), set(m.text_id)
    if st != sm:
        print(f"  PERINGATAN {model}: teks test beda (text-only {len(st)}, meta {len(sm)}, irisan {len(st & sm)})."
              " Evaluasi memakai irisan.")
    common = st & sm
    t, m = t[t.text_id.isin(common)], m[m.text_id.isin(common)]

    # level annotator: prediksi text-only di-broadcast ke semua annotator teksnya
    ann = m.merge(t[["text_id", "prob", "y_true"]].rename(columns={"prob": "prob_text", "y_true": "majority"}),
                  on="text_id").rename(columns={"prob": "prob_meta"})
    ann["pred_text"] = (ann.prob_text >= 0.5).astype(int)
    ann["pred_meta"] = (ann.prob_meta >= 0.5).astype(int)

    # level teks: rata-rata probabilitas metadata per teks
    txt = (m.groupby("text_id")["prob"].mean().rename("prob_meta").reset_index()
           .merge(t.rename(columns={"prob": "prob_text"}), on="text_id"))
    txt["pred_text"] = (txt.prob_text >= 0.5).astype(int)
    txt["pred_meta"] = (txt.prob_meta >= 0.5).astype(int)
    return ann, txt


rows = []
for model in MODELS:
    ann, txt = stage1_frames(model)
    pref = f"{model} | S1"
    for key in ["f1_toxic", "f1_macro"]:
        rows.append({"model": model, **summarize(f"{pref} | level annotator", ann, "y_true", "pred_text", "pred_meta", bin_metrics, key)})
        rows.append({"model": model, **summarize(f"{pref} | level teks", txt, "y_true", "pred_text", "pred_meta", bin_metrics, key)})

    # subset >=2 annotator, teks sengketa & baris minoritas
    stat = ann.groupby("text_id")["y_true"].agg(["count", "nunique"])
    multi = ann[ann.text_id.isin(stat[stat["count"] >= 2].index)]
    for key in ["f1_toxic", "f1_macro"]:
        rows.append({"model": model, **summarize(f"{pref} | teks >=2 annotator (annotator)", multi, "y_true", "pred_text", "pred_meta", bin_metrics, key)})
    disputed = stat[(stat["count"] >= 2) & (stat["nunique"] > 1)].index
    d = ann[ann.text_id.isin(disputed)]
    minority = d[d.y_true != d.majority]
    print(f"{model}: {len(disputed)} teks sengketa di test, {len(minority)} baris minoritas")
    if len(d):
        rows.append({"model": model, **summarize(f"{pref} | teks sengketa (annotator)", d, "y_true", "pred_text", "pred_meta", bin_metrics, "f1_macro")})
    if len(minority):
        rows.append({"model": model, **summarize(f"{pref} | baris minoritas", minority, "y_true", "pred_text", "pred_meta", bin_metrics, "accuracy")})

res_s1 = pd.DataFrame(rows)
res_s1

**Cara baca:** `selisih` = metadata − text-only. `signifikan = ya` kalau 95% CI tidak melewati 0.
Untuk *baris minoritas*, text-only hampir selalu menebak label mayoritas, jadi akurasinya rendah secara konstruksi; yang menarik adalah seberapa jauh metadata bisa menaikkannya.

### Per kelompok demografis (level annotator, Stage 1)

In [ ]:
group_rows = []
for model in MODELS:
    ann, _ = stage1_frames(model)
    for col in [c for c in DEMO_COLS if c in ann.columns]:
        for val, g in ann.groupby(col):
            if g["y_true"].nunique() < 2 or len(g) < 30:
                continue
            group_rows.append({"model": model, "atribut": col, "nilai": val, "n_baris": len(g),
                               "f1_macro_text": round(f1_score(g.y_true, g.pred_text, average="macro"), 4),
                               "f1_macro_meta": round(f1_score(g.y_true, g.pred_meta, average="macro"), 4)})
res_group = pd.DataFrame(group_rows)
res_group["selisih"] = res_group["f1_macro_meta"] - res_group["f1_macro_text"]
res_group.sort_values(["model", "atribut", "nilai"])

### Per seed (kalau `FINAL_SEEDS` lebih dari satu)

In [ ]:
seed_rows = []
for model in MODELS:
    seeds = sorted(set(load(model, "text", 1).seed) & set(load(model, "meta", 1).seed))
    for s in seeds:
        ann, txt = stage1_frames(model, seed=s)
        for lvl, d in [("annotator", ann), ("teks", txt)]:
            seed_rows.append({"model": model, "level": lvl, "seed": s,
                              "f1_macro_text": f1_score(d.y_true, d.pred_text, average="macro"),
                              "f1_macro_meta": f1_score(d.y_true, d.pred_meta, average="macro")})
seed_df = pd.DataFrame(seed_rows)
if seed_df.groupby(["model", "level"]).size().max() > 1:
    display(seed_df.groupby(["model", "level"])[["f1_macro_text", "f1_macro_meta"]].agg(["mean", "std"]).round(4))
else:
    print("Hanya 1 seed per model; tabel mean ± std dilewati.")

## Stage 2 (jenis toksisitas)

In [ ]:
TRUE = [f"true_{l}" for l in LABEL_COLS]
PROB = [f"prob_{l}" for l in LABEL_COLS]

def stage2_frames(model):
    t, m = load(model, "text", 2), load(model, "meta", 2)
    t = t.groupby("text_id")[TRUE + PROB].agg({**{c: "first" for c in TRUE}, **{c: "mean" for c in PROB}}).reset_index()
    m = m.groupby(["text_id", "annotator_id"])[TRUE + PROB].agg({**{c: "first" for c in TRUE}, **{c: "mean" for c in PROB}}).reset_index()
    common = set(t.text_id) & set(m.text_id)
    if len(common) < t.text_id.nunique():
        print(f"  {model}: {t.text_id.nunique() - len(common)} teks test text-only tidak punya baris annotator toxic di metadata")
    t, m = t[t.text_id.isin(common)], m[m.text_id.isin(common)]

    ann = m.merge(t[["text_id"] + PROB].rename(columns={p: p + "_text" for p in PROB}), on="text_id")
    txt = (m.groupby("text_id")[PROB].mean().reset_index()
           .merge(t.rename(columns={p: p + "_text" for p in PROB}), on="text_id"))
    return ann, txt


def ml_stat(d, suffix):
    P = (d[[p + suffix for p in PROB]].values >= 0.5).astype(int)
    return ml_macro_f1(d[TRUE].values, P)


rows2 = []
for model in MODELS:
    ann2, txt2 = stage2_frames(model)
    n_ann = ann2.groupby("text_id")["annotator_id"].transform("count")
    for lvl, d in [("level annotator", ann2), ("level teks", txt2),
                   ("teks >=2 annotator (annotator)", ann2[n_ann >= 2].reset_index(drop=True))]:
        a, b = ml_stat(d, "_text"), ml_stat(d, "")
        lo, hi = cluster_bootstrap(d, lambda x: ml_stat(x, "") - ml_stat(x, "_text"))
        rows2.append({"model": model, "evaluasi": f"{model} | S2 | {lvl}", "metrik": "macro_f1",
                      "n_baris": len(d), "n_teks": d.text_id.nunique(),
                      "text_only": round(a, 4), "metadata": round(b, 4), "selisih": round(b - a, 4),
                      "ci_low": round(lo, 4), "ci_high": round(hi, 4),
                      "signifikan": "ya" if (lo > 0 or hi < 0) else "tidak"})
res_s2 = pd.DataFrame(rows2)
res_s2

In [ ]:
results = pd.concat([res_s1, res_s2], ignore_index=True)
results.to_csv(OUT_FILE, index=False)
res_group.to_csv(OUT_FILE.replace(".csv", "_per_group.csv"), index=False)
print("Tersimpan:", OUT_FILE)
results